# Nigeria Immunization Equity & Zero-Dose Risk Stratification
## NPHCDA Digital Innovation Hub

**Principal Investigator:** Dr. Amobi Andrew Onovo, PhD, MPH  
**Consortium:** CIDRE–Quantium Insights

### Research question
How does predicted immunization coverage and zero-dose risk vary by location (rural vs. urban), poverty level, maternal education, and hard-to-reach (HTR) status?

### Analytical strategy

The question is answered with two complementary spatial methods. The phrase "by location" in the research question is treated literally: the analysis must quantify how the effect of each equity dimension varies across the country, not just estimate a single national effect.

**Approach A — Multiscale Geographically Weighted Regression (MGWR).**
A spatial regression in which the coefficient on each equity dimension is allowed to vary across space, with each predictor receiving its own optimal bandwidth selected by corrected Akaike Information Criterion (AICc). The output is a coefficient *per state per predictor*, mapped across the country, showing where each equity dimension matters most.

**Approach B — Composite equity-deprivation index.**
The four equity dimensions are rescaled to a 0–100 range and combined into a single composite score per state, which is classified into four programmatic tiers. The output is a ranked state list intended for direct operational use.

The two approaches answer distinct questions: Approach A is diagnostic and spatial — *where does each dimension matter?* Approach B is operational — *which states have the worst combined profile?*

**Outcome.** Children 12–23 months reported as not having received DTP1 by the end of their first year of life (NDHS 2024).

**Equity dimensions (NDHS / Living Standards Survey microdata, aggregated to state).**
- Urbanicity — percent of population in urban localities
- Hard-to-reach status — percent of women reporting distance to health facility as a problem
- Maternal education — percent of women with no formal education
- Poverty — percent of households in the lowest national wealth quintile


---

In [1]:
# Detect whether running in Google Colab or locally
try:
    import google.colab
    IN_COLAB = True
    print("Environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environment: Local Jupyter / JupyterLab")


Environment: Google Colab


In [2]:
# Install required packages — run once per session
# In Colab this takes ~90 seconds on first run

import subprocess, sys

packages = [
    "mgwr", "libpysal", "spglm",     # spatial regression stack
    "geopandas",                       # optional, used if GRID3 shapefiles uploaded
    "scipy", "scikit-learn",
    "matplotlib", "pandas", "numpy",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Packages installed.")


Packages installed.


### Upload data files

Two uploads are needed (the second is optional):

1. **Required:** `nigeria_equity_indicators.csv` — state-level NDHS 2024 indicators (state, zone, zero-dose rate, four equity dimensions).
2. **Optional:** `spatial.zipx` — GRID3 admin1 (state) shapefile bundle. When supplied, choropleth maps use proper polygon boundaries. When omitted, point-symbol maps at state centroids are used instead.


In [8]:
# ─── COLAB: Upload data file(s) ─────────────────────────────────────────
# Required:
#   nigeria_equity_indicators.csv
# Optional:
#   spatial.zipx  (GRID3 admin1 boundaries for state-level choropleth)

if IN_COLAB:
    from google.colab import files
    print("Select nigeria_equity_indicators.csv (and optionally spatial.zipx)...")
    uploaded = files.upload()
    for f in uploaded:
        print(f"Uploaded: {f}  ({len(uploaded[f])/1024:.0f} KB)")
else:
    import os
    expected = ["nigeria_equity_indicators.csv"]
    optional = ["spatial.zipx", "spatial.zip"]
    for f in expected:
        status = "OK" if os.path.exists(f) else "MISSING"
        print(f"  {status}  {f}")
    for f in optional:
        if os.path.exists(f):
            print(f"  OK   {f}  (optional)")


Select nigeria_equity_indicators.csv (and optionally spatial.zipx)...


Saving nigeria_equity_indicators.csv to nigeria_equity_indicators.csv
Saving spatial.zipx to spatial (1).zipx
Uploaded: nigeria_equity_indicators.csv  (7 KB)
Uploaded: spatial (1).zipx  (6566 KB)


In [4]:
# Extract spatial archive if uploaded — supports .zip and .zipx (uses system 7z)

import os, subprocess, glob

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

archives = glob.glob("spatial.zipx") + glob.glob("spatial.zip")
HAVE_SHAPEFILES = False

if archives:
    archive = archives[0]
    print(f"Found archive: {archive}")

    if IN_COLAB:
        _run("apt-get install -y p7zip-full -q 2>/dev/null")

    rc, out = _run(f'7z x "{archive}" -o./ -y 2>&1 | tail -3')
    if rc == 0:
        print("Extraction OK.")
    else:
        print(f"Extraction failed: {out[:300]}")

    # Locate the state shapefile under any plausible path
    candidates = (glob.glob("**/grid3_nga_boundary_vaccstates.shp", recursive=True) +
                  glob.glob("**/*admin1*.shp", recursive=True) +
                  glob.glob("**/*state*.shp", recursive=True))
    if candidates:
        PATH_SHP_STATE = candidates[0]
        HAVE_SHAPEFILES = True
        print(f"State shapefile: {PATH_SHP_STATE}")
    else:
        print("Archive extracted but no state shapefile found.")
        PATH_SHP_STATE = None
else:
    PATH_SHP_STATE = None
    print("No spatial archive — will use point-symbol maps at state centroids.")


Found archive: spatial.zipx
Extraction OK.
State shapefile: admin1/grid3_nga_boundary_vaccstates.shp


---

In [5]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")        # change to "inline" for Jupyter display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW
from scipy import stats

# Geopandas is loaded lazily — only needed if shapefiles uploaded
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except ImportError:
    HAVE_GEOPANDAS = False

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─── CONFIGURATION ──────────────────────────────────────────────────────
PATH_EQUITY = "nigeria_equity_indicators.csv"

OUTPUT_DIR = "outputs"

# MGWR settings — adaptive bisquare kernel with corrected-AIC bandwidth selection
KERNEL          = "bisquare"
BW_MIN          = 6                 # smallest neighbourhood (must exceed k+1 = 5)
ALPHA_LOCAL_SIG = 0.05              # significance threshold for local t-tests

# Composite-index weights (must sum to 1.0)
W_RURAL, W_HTR, W_EDU, W_WLTH = 0.25, 0.25, 0.25, 0.25

# Map output resolution — 300 DPI is the standard for print-journal submissions
MAP_DPI = 300

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Configuration ready. Output directory: {OUTPUT_DIR}")


Configuration ready. Output directory: outputs


---

In [6]:
# State centroids (decimal degrees, WGS84) — fallback only.
# When GRID3 shapefiles are uploaded, centroids are computed from the polygons.
FALLBACK_CENTROIDS = {
    "Abia":          (5.4527,  7.5248),  "Adamawa":      (9.3265, 12.3984),
    "Akwa Ibom":     (4.9057,  7.8537),  "Anambra":      (6.2209,  6.9370),
    "Bauchi":       (10.7760,  9.9990),  "Bayelsa":      (4.7719,  6.0699),
    "Benue":         (7.1907,  8.1290),  "Borno":       (11.8333, 13.1500),
    "Cross River":   (5.8702,  8.5988),  "Delta":        (5.7040,  5.9339),
    "Ebonyi":        (6.2649,  8.0137),  "Edo":          (6.6342,  5.9304),
    "Ekiti":         (7.7190,  5.3110),  "Enugu":        (6.5244,  7.5086),
    "Fct":           (8.8940,  7.1860),  "Gombe":       (10.2904, 11.1700),
    "Imo":           (5.4951,  7.0260),  "Jigawa":      (12.5800,  9.7800),
    "Kaduna":       (10.5105,  7.4165),  "Kano":        (12.0000,  8.5167),
    "Katsina":      (12.9908,  7.6018),  "Kebbi":       (12.4500,  4.1995),
    "Kogi":          (7.8000,  6.7400),  "Kwara":        (8.5000,  4.5500),
    "Lagos":         (6.5244,  3.3792),  "Nasarawa":     (8.5705,  8.2987),
    "Niger":        (10.0000,  5.4000),  "Ogun":         (7.1606,  3.3486),
    "Ondo":          (7.0989,  5.0750),  "Osun":         (7.5629,  4.5200),
    "Oyo":           (7.8500,  3.9333),  "Plateau":      (9.2182,  9.5179),
    "Rivers":        (4.7500,  6.8333),  "Sokoto":      (13.0059,  5.2476),
    "Taraba":        (8.8930, 11.3600),  "Yobe":        (12.0000, 11.5000),
    "Zamfara":      (12.1700,  6.6600),
}

def normalise_name(s):
    """Standardise state/zone name for joining."""
    return str(s).strip().title().replace("-", " ").replace("_", " ")

def minmax_scale(x):
    """Scale a series to a 0–100 range."""
    r = x.max() - x.min()
    return (x - x.min()) / r * 100 if r > 0 else x * 0 + 50

# ─── Cartographic helpers — used by all map figures ──────────────────
def add_scale_bar(ax, length_km=200, location=(0.05, 0.05),
                   color="#222", lat_at=9.0):
    """Scale bar in decimal degrees, accurate at the given latitude."""
    deg_per_km = 1.0 / (111.32 * np.cos(np.radians(lat_at)))
    bar_len = length_km * deg_per_km
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    x0 = xlim[0] + (xlim[1]-xlim[0]) * location[0]
    y0 = ylim[0] + (ylim[1]-ylim[0]) * location[1]
    x1 = x0 + bar_len
    ax.plot([x0, x1], [y0, y0], color=color, lw=2.2,
            solid_capstyle="butt", zorder=20)
    for x in [x0, x1]:
        ax.plot([x, x], [y0, y0 + (ylim[1]-ylim[0])*0.008],
                color=color, lw=1.6, zorder=20)
    ax.text((x0+x1)/2, y0 + (ylim[1]-ylim[0])*0.020,
            f"{length_km} km", ha="center", va="bottom",
            fontsize=7.5, color=color, zorder=20)

def add_north_arrow(ax, location=(0.95, 0.94), size=0.07, color="#222"):
    """Triangle north arrow."""
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    cx = xlim[0] + (xlim[1]-xlim[0]) * location[0]
    cy = ylim[0] + (ylim[1]-ylim[0]) * location[1]
    h  = (ylim[1]-ylim[0]) * size
    w  = h * 0.32
    tri = plt.Polygon([(cx, cy+h/2), (cx-w/2, cy-h/2), (cx+w/2, cy-h/2)],
                       closed=True, color=color, zorder=20)
    ax.add_patch(tri)
    ax.text(cx, cy + h*0.62, "N", ha="center", va="bottom",
            fontsize=9.5, fontweight="bold", color=color, zorder=20)

def label_states(ax, gdf_subset, fontsize=7.5, weight="bold", color="#111"):
    """Halo-style state labels at polygon representative points."""
    for _, r in gdf_subset.iterrows():
        c = r.geometry.representative_point()
        t = ax.annotate(r["state"], (c.x, c.y),
                        ha="center", va="center",
                        fontsize=fontsize, fontweight=weight, color=color,
                        zorder=15)
        t.set_path_effects([
            path_effects.withStroke(linewidth=2.4, foreground="white")
        ])

def style_map_axes(ax):
    """Common axis styling for choropleth maps."""
    ax.set_xticks(np.arange(4, 15, 2))
    ax.set_yticks(np.arange(5, 14, 2))
    ax.tick_params(axis="both", labelsize=7.5, color="#888",
                    length=3, pad=2)
    ax.set_xlabel("Longitude (°E)", fontsize=8.5, color="#444")
    ax.set_ylabel("Latitude (°N)", fontsize=8.5, color="#444")
    for s in ax.spines.values():
        s.set_color("#AAA"); s.set_linewidth(0.6)
    ax.set_aspect("equal")

# Palette: aligned with the broader zero-dose modelling work
TIER_COLORS = {
    "Tier 1: Most disadvantaged": "#D73027",
    "Tier 2: Disadvantaged":      "#FC8D59",
    "Tier 3: Moderate":           "#FEE090",
    "Tier 4: Most equitable":     "#91BFDB",
}
ZONE_COLORS = {
    "North West":   "#D73027", "North East":  "#FC8D59",
    "North Central":"#FDAE61", "South West":  "#1A9850",
    "South East":   "#91CF60", "South South": "#66C2A5",
}
QUINT_COLORS = ["#1A9850","#91CF60","#FEE090","#FC8D59","#D73027"]
BG = "#F8F9FA"

EQUITY_VARS = ["urbanicity", "htr_distance", "no_education", "low_wealth"]
EQUITY_LABELS = {
    "urbanicity":   "Urbanicity (% urban)",
    "htr_distance": "HTR distance (% reporting access problem)",
    "no_education": "Maternal no-education (%)",
    "low_wealth":   "Lowest wealth quintile (%)",
}

print("Helpers, cartographic functions, and palette defined.")


Helpers, cartographic functions, and palette defined.


## 1. Load and inspect the equity-indicator dataset

State-level NDHS 2024 indicators are joined to the four-dimension equity profile, and state centroids (decimal degrees, WGS84) are attached for use as observation locations in MGWR. The dataset includes 37 states (36 + FCT) with no missing values on any equity dimension.


In [9]:
raw = pd.read_csv(PATH_EQUITY)
raw = raw.loc[:, ~raw.columns.duplicated()]

df = raw.rename(columns={
    "state_name":                       "state",
    "zone_name":                        "zone",
    "zero_dose_2024":                   "zd_rate",
    "pct_urban":                        "urbanicity",
    "pct_problem_accessing_hfdistance": "htr_distance",
    "pct_women_no_education":           "no_education",
    "pct_lowest_wealth_quintile":       "low_wealth",
})

# Keep all available columns for any downstream use
df["state"] = df["state"].apply(normalise_name)
df["zone"]  = df["zone"].apply(normalise_name)

# ─── Load shapefiles if available; otherwise fall back to centroids ────
GEO_AVAILABLE = False
gdf_state = None
gdf_lga   = None
geo       = None
boundary  = None

if HAVE_SHAPEFILES and HAVE_GEOPANDAS:
    gdf_state = gpd.read_file(PATH_SHP_STATE)
    gdf_state["state"] = gdf_state["statename"].apply(normalise_name)
    # Try to find LGA shapefile alongside the state shapefile
    import glob
    lga_cands = (glob.glob("**/grid3_nga_boundary_vacclgas.shp", recursive=True)
                 + glob.glob("**/*admin2*.shp", recursive=True))
    if lga_cands:
        gdf_lga = gpd.read_file(lga_cands[0])
        print(f"LGA shapefile: {lga_cands[0]}  ({len(gdf_lga)} LGAs)")
    # Merge data with geometries
    geo = gdf_state.merge(df, on="state", how="left")
    # Compute centroids in a projected CRS for accuracy, then project back to WGS84
    geo_proj = geo.to_crs("EPSG:32632")          # UTM zone 32N covers western Nigeria
    cents = geo_proj.geometry.centroid.to_crs(epsg=4326)
    df_idx = geo[["state"]].reset_index(drop=True)
    df_idx["lon"] = cents.x.values
    df_idx["lat"] = cents.y.values
    df = df.merge(df_idx, on="state", how="left")
    boundary = geo.dissolve().boundary
    GEO_AVAILABLE = True
    print(f"GRID3 polygon mode: matched {geo['zd_rate'].notna().sum()}/{len(geo)} states.")
else:
    df["lat"] = df["state"].map(lambda s: FALLBACK_CENTROIDS.get(s, (np.nan, np.nan))[0])
    df["lon"] = df["state"].map(lambda s: FALLBACK_CENTROIDS.get(s, (np.nan, np.nan))[1])
    print("Centroid-only mode (no shapefiles). Maps will use point symbols.")

missing_geo = df[df["lat"].isna()]["state"].tolist()
if missing_geo:
    print(f"WARNING — no centroid for: {missing_geo}")

print()
print(f"States loaded : {len(df)}")
print(f"Zones         : {df['zone'].nunique()}")
print()
print("Outcome and equity dimensions — descriptive statistics:")
df[["zd_rate"] + EQUITY_VARS].describe().round(1)


LGA shapefile: admin2/grid3_nga_boundary_vacclgas.shp  (774 LGAs)
GRID3 polygon mode: matched 37/37 states.

States loaded : 37
Zones         : 6

Outcome and equity dimensions — descriptive statistics:


,zd_rate,urbanicity,htr_distance,no_education,low_wealth
count,37.0,37.0,37.0,37.0,37.0
mean,29.5,47.4,25.4,28.1,16.8
std,22.9,25.0,12.8,28.6,19.0
min,2.6,9.8,2.9,0.5,0.0
25%,9.4,25.6,18.1,3.3,1.0
50%,29.1,42.9,23.7,11.6,10.5
75%,40.4,67.2,31.5,53.0,24.4
max,86.5,98.7,60.0,85.8,68.4


## 1A. Observed zero-dose baseline (NDHS 2024)

The map below shows the spatial pattern of observed zero-dose burden across the 37 states, drawn on GRID3 admin1 boundaries with LGA underlay. This is the dependent variable the MGWR will explain. Naming the highest-burden cluster up-front frames everything that follows.

In [10]:
# Figure 1 — Observed zero-dose rate by state (premium choropleth)
if GEO_AVAILABLE:
    fig, ax = plt.subplots(figsize=(10, 9), facecolor="white")
    xmin, ymin, xmax, ymax = geo.total_bounds
    pad = 0.5
    ax.set_xlim(xmin - pad, xmax + pad); ax.set_ylim(ymin - pad, ymax + pad + 0.3)
    ax.set_facecolor("#E8EFF5")

    if gdf_lga is not None:
        gdf_lga.plot(ax=ax, facecolor="none", edgecolor="#888",
                      lw=0.13, alpha=0.30, zorder=2)

    geo.plot(column="zd_rate", cmap="YlOrRd", ax=ax,
              edgecolor="white", lw=0.55, zorder=3,
              legend=True,
              legend_kwds={"label":"Zero-dose rate (% of children 12–23 mo, NDHS 2024)",
                            "shrink":0.65, "pad":0.02, "aspect":25})
    boundary.plot(ax=ax, color="#111", lw=1.2, zorder=5)

    # Annotate the highest and lowest burden states
    top = geo.nlargest(8, "zd_rate")
    bot = geo.nsmallest(3, "zd_rate")
    label_states(ax, pd.concat([top, bot]))

    add_north_arrow(ax); add_scale_bar(ax)
    style_map_axes(ax)
    ax.set_title("Observed zero-dose rate by state, Nigeria 2024\n"
                  "Children 12–23 months who did not receive DTP1 (NDHS 2024)",
                  fontsize=13, fontweight="bold", pad=10, color="#1F3864")
    fig.text(0.5, 0.02,
              "Boundaries: GRID3 Nigeria admin1 (vaccination layer) and admin2 (LGAs, faint).  "
              "Data: NDHS 2024.  Projection: WGS84 / EPSG:4326.",
              ha="center", fontsize=8, color="#666", style="italic")
    plt.tight_layout(rect=[0, 0.04, 1, 0.99])
else:
    # Fallback scatter
    fig, ax = plt.subplots(figsize=(10, 8), facecolor="white")
    ax.set_facecolor(BG)
    sc = ax.scatter(df["lon"], df["lat"], c=df["zd_rate"], cmap="YlOrRd",
                     s=320, edgecolor="#222", lw=0.6, alpha=0.92)
    plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02,
                  label="Zero-dose rate (%)")
    for _, r in df.iterrows():
        ax.annotate(r["state"], (r["lon"], r["lat"]),
                     textcoords="offset points", xytext=(5,5), fontsize=7)
    ax.set_title("Observed zero-dose rate by state, Nigeria 2024",
                  fontsize=13, fontweight="bold", pad=10, color="#1F3864")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

plt.savefig(f"{OUTPUT_DIR}/fig01_observed_zd_choropleth.png", dpi=MAP_DPI,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig01_observed_zd_choropleth.png")


Saved: fig01_observed_zd_choropleth.png


## 2. Approach A — Multiscale Geographically Weighted Regression (MGWR)

### Why this method

A conventional regression assumes that the effect of each predictor is constant across space — a single slope coefficient describes the relationship for every state. The research question explicitly asks how zero-dose risk varies *by location*, which implies that the effect of each equity dimension is allowed to differ across the country. The standard tool for this in the spatial-statistics literature is Geographically Weighted Regression (Brunsdon, Fotheringham & Charlton 1996).

GWR fits a separate regression at each location, weighting nearby observations more heavily than distant ones. The result is one set of coefficients per location, mapped across the study area. The single-bandwidth GWR has one limitation: it assumes that every predictor varies at the same spatial scale. **Multiscale GWR (MGWR)** relaxes this assumption — each predictor receives its own optimal bandwidth, so dimensions that vary smoothly across the country (e.g. urbanicity) get a large bandwidth while dimensions that change abruptly (e.g. access to facilities in conflict zones) get a small bandwidth. This is published in Fotheringham, Yang & Kang (2017).

### Model specification

For each state *s* at location *(x_s, y_s)*:

    zd_rate[s] = β₀(s) + β_urban(s)·urbanicity_z[s] + β_htr(s)·htr_z[s]
                       + β_edu(s)·no_education_z[s] + β_wlth(s)·low_wealth_z[s] + ε[s]

Each coefficient β_k(s) is estimated via locally weighted least squares using a bisquare kernel:

    w(d_sj) = (1 − (d_sj / bw_k)²)²    for d_sj < bw_k, else 0

The bandwidth `bw_k` is *adaptive* (specified as a number of nearest neighbours rather than a fixed distance), which is the appropriate choice when observation density varies across the study area. Each predictor `k` receives its own bandwidth, selected to minimise the corrected Akaike Information Criterion (AICc) — the small-sample-corrected AIC, standard for n < 50.

### Why AICc

For n = 37 states the standard AIC overfits noticeably. AICc adds a penalty proportional to `2k(k+1)/(n−k−1)` which shrinks toward zero for large n but is non-trivial here. AICc is the published default in the MGWR literature for small spatial samples.

### Estimation procedure

1. Fit single-bandwidth GWR with AICc to obtain initial bandwidth.
2. MGWR backfitting: iteratively refit each predictor's bandwidth and coefficient surface, holding others fixed, until convergence.
3. Compute local t-statistics for each coefficient.
4. Apply Bonferroni-style correction for the multiple local tests to obtain the adjusted critical |t| value.


In [11]:
# Build the matrices MGWR expects: y (n,1), X (n,k), coords list of (x, y)
y_mat   = df["zd_rate"].values.reshape(-1, 1)
X_mat   = df[EQUITY_VARS].values
scaler  = StandardScaler()
X_z     = scaler.fit_transform(X_mat)
coords  = list(zip(df["lon"].values, df["lat"].values))
n_obs   = len(df)

print(f"n = {n_obs}, k = {X_z.shape[1]} predictors")
print(f"Coords: x range [{min(c[0] for c in coords):.2f}, {max(c[0] for c in coords):.2f}], "
      f"y range [{min(c[1] for c in coords):.2f}, {max(c[1] for c in coords):.2f}]")

# Global OLS baseline (no spatial variation) — for comparison
ols       = LinearRegression().fit(X_z, y_mat.ravel())
ols_pred  = ols.predict(X_z)
ols_resid = y_mat.ravel() - ols_pred
ols_r2    = 1 - (ols_resid ** 2).sum() / ((y_mat.ravel() - y_mat.mean()) ** 2).sum()
print(f"\nGlobal OLS R² (baseline): {ols_r2:.3f}")
print(f"OLS coefficients (z-scored predictors):")
for v, b in zip(EQUITY_VARS, ols.coef_):
    print(f"  {v:14s} = {b:+.2f}")


n = 37, k = 4 predictors
Coords: x range [3.47, 13.16], y range [4.77, 13.06]

Global OLS R² (baseline): 0.743
OLS coefficients (z-scored predictors):
  urbanicity     = -2.88
  htr_distance   = +0.81
  no_education   = +16.05
  low_wealth     = -4.32


In [12]:
# Step 1 — single-bandwidth GWR with AICc-adaptive bandwidth selection
print("Searching single-bandwidth GWR (AICc, adaptive bisquare)...")
gwr_selector = Sel_BW(coords, y_mat, X_z, fixed=False, kernel=KERNEL)
gwr_bw       = gwr_selector.search(criterion="AICc", bw_min=BW_MIN, bw_max=n_obs - 1)
print(f"Optimal GWR bandwidth: {int(gwr_bw)} nearest neighbours")

gwr_model = GWR(coords, y_mat, X_z, gwr_bw, fixed=False, kernel=KERNEL).fit()
print(f"GWR R²: {gwr_model.R2:.3f}   adj R²: {gwr_model.adj_R2:.3f}   AICc: {gwr_model.aicc:.2f}")


Searching single-bandwidth GWR (AICc, adaptive bisquare)...
Optimal GWR bandwidth: 35 nearest neighbours
GWR R²: 0.821   adj R²: 0.755   AICc: 303.21


In [13]:
# Step 2 — MGWR: each predictor gets its own bandwidth via AICc backfitting
print("Searching MGWR bandwidths (this may take a minute)...")
mgwr_selector = Sel_BW(coords, y_mat, X_z, multi=True, fixed=False, kernel=KERNEL)
mgwr_bws      = mgwr_selector.search(
    criterion   = "AICc",
    init_multi  = gwr_bw,                   # warm-start from single-bw GWR
    multi_bw_min= [BW_MIN] * (X_z.shape[1] + 1),
    multi_bw_max= [n_obs - 1] * (X_z.shape[1] + 1),
)
print(f"MGWR bandwidths per predictor:")
for name, bw in zip(["intercept"] + EQUITY_VARS, mgwr_bws):
    print(f"  {name:14s} = {int(bw):3d} neighbours")

mgwr_model = MGWR(coords, y_mat, X_z, mgwr_selector, fixed=False, kernel=KERNEL).fit()
print(f"\nMGWR fit: R² = {mgwr_model.R2:.3f}   adj R² = {mgwr_model.adj_R2:.3f}   AICc = {mgwr_model.aicc:.2f}")


Searching MGWR bandwidths (this may take a minute)...


Backfitting:   0%|          | 0/200 [00:00<?, ?it/s]

MGWR bandwidths per predictor:
  intercept      =  35 neighbours
  urbanicity     =  24 neighbours
  htr_distance   =  35 neighbours
  no_education   =  28 neighbours
  low_wealth     =  27 neighbours


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


MGWR fit: R² = 0.868   adj R² = 0.803   AICc = 301.65


In [14]:
# Step 3 — compare the three model fits
comparison = pd.DataFrame({
    "Model":  ["Global OLS (no spatial variation)",
               f"GWR (single bandwidth = {int(gwr_bw)})",
               f"MGWR (per-predictor bandwidths)"],
    "R2":     [round(ols_r2, 3),       round(gwr_model.R2, 3),  round(mgwr_model.R2, 3)],
    "adj_R2": [None,                    round(gwr_model.adj_R2, 3), round(mgwr_model.adj_R2, 3)],
    "AICc":   [None,                    round(gwr_model.aicc, 2),  round(mgwr_model.aicc, 2)],
})
print(comparison.to_string(index=False))
comparison.to_csv(f"{OUTPUT_DIR}/equity_model_comparison.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/equity_model_comparison.csv")


                            Model    R2  adj_R2   AICc
Global OLS (no spatial variation) 0.743     NaN    NaN
      GWR (single bandwidth = 35) 0.821   0.755 303.21
  MGWR (per-predictor bandwidths) 0.868   0.803 301.65

Saved: outputs/equity_model_comparison.csv


In [15]:
# Step 4 — per-state coefficients from MGWR (intercept + four equity slopes)
mgwr_params = mgwr_model.params                # (n, k+1)
mgwr_t      = mgwr_model.tvalues
crit_t      = mgwr_model.critical_tval(alpha=ALPHA_LOCAL_SIG)  # Bonferroni-style
print(f"Bonferroni-style critical |t| at alpha={ALPHA_LOCAL_SIG}: {crit_t:.3f}")

coef_cols = ["intercept", "b_urban", "b_htr", "b_edu", "b_wlth"]
mgwr_coef = pd.DataFrame(mgwr_params, columns=coef_cols)
mgwr_tval = pd.DataFrame(mgwr_t,      columns=[f"t_{c[2:] if c.startswith('b_') else c}" for c in coef_cols])
mgwr_sig  = pd.DataFrame((np.abs(mgwr_t) > crit_t).astype(int),
                          columns=[f"sig_{c[2:] if c.startswith('b_') else c}" for c in coef_cols])

mgwr_full = pd.concat([df[["state","zone","lat","lon","zd_rate"]].reset_index(drop=True),
                       mgwr_coef, mgwr_tval, mgwr_sig], axis=1)

# Summary of per-predictor coefficient distributions
print("\nMGWR local coefficient distributions:")
print(f"{'predictor':<14s}{'min':>8s}{'25th':>8s}{'median':>8s}{'75th':>8s}{'max':>8s}{'range':>8s}{'n_sig':>7s}")
for col, sig_col, name in zip(coef_cols[1:],
                                ["sig_urban","sig_htr","sig_edu","sig_wlth"],
                                EQUITY_VARS):
    p = mgwr_coef[col].values
    n_sig = mgwr_sig[sig_col].sum()
    print(f"{name:<14s}{p.min():>8.2f}{np.percentile(p,25):>8.2f}{np.median(p):>8.2f}"
          f"{np.percentile(p,75):>8.2f}{p.max():>8.2f}{p.max()-p.min():>8.2f}{n_sig:>5d}/37")

mgwr_full.to_csv(f"{OUTPUT_DIR}/equity_mgwr_state_coefficients.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/equity_mgwr_state_coefficients.csv")


Bonferroni-style critical |t| at alpha=0.05: 2.028

MGWR local coefficient distributions:
predictor          min    25th  median    75th     max   range  n_sig
urbanicity       -8.59   -4.50   -3.02   -1.43    0.47    9.07    3/37
htr_distance     -0.12    0.08    0.32    0.54    1.58    1.70    0/37
no_education      8.22   14.57   17.90   19.25   20.57   12.34   37/37
low_wealth       -3.97   -3.30   -2.16   -1.42    1.06    5.03    0/37

Saved: outputs/equity_mgwr_state_coefficients.csv


In [16]:
# Figure 2 — MGWR coefficient surfaces (premium choropleth maps)
if GEO_AVAILABLE:
    # Attach MGWR coefficients & significance to the geo dataframe
    for i, nm in enumerate(["intercept","b_urban","b_htr","b_edu","b_wlth"]):
        geo[nm] = mgwr_params[:, i]
    for i, nm in enumerate(["sig_intercept","sig_urban","sig_htr","sig_edu","sig_wlth"]):
        geo[nm] = (np.abs(mgwr_t[:, i]) > crit_t).astype(int)
    # Also attach predicted ZD for joint synthesis later
    geo["zd_pred"] = mgwr_model.predy.ravel()

    def _draw_coef_panel(ax, coef_col, sig_col, name):
        xmin, ymin, xmax, ymax = geo.total_bounds
        pad = 0.5
        ax.set_xlim(xmin - pad, xmax + pad)
        ax.set_ylim(ymin - pad, ymax + pad + 0.3)
        ax.set_facecolor("#E8EFF5")

        if gdf_lga is not None:
            gdf_lga.plot(ax=ax, facecolor="none", edgecolor="#888",
                         lw=0.12, alpha=0.28, zorder=2)

        vals = geo[coef_col].values
        vmax = max(abs(np.nanmin(vals)), abs(np.nanmax(vals)))
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

        sig_mask = geo[sig_col] == 1
        n_sig = sig_mask.sum()

        if n_sig == 0:
            # Show muted gradient + hatching to indicate no significance
            geo.plot(column=coef_col, cmap="RdBu_r", norm=norm, ax=ax,
                     edgecolor="white", lw=0.45, zorder=3, alpha=0.5)
            geo.plot(facecolor="none", edgecolor="#888", hatch="///",
                     lw=0.20, ax=ax, zorder=4, alpha=0.35)
        else:
            sig_geo = geo[sig_mask]; non_geo = geo[~sig_mask]
            sig_geo.plot(column=coef_col, cmap="RdBu_r", norm=norm, ax=ax,
                         edgecolor="white", lw=0.55, zorder=3)
            if not non_geo.empty:
                non_geo.plot(facecolor="#E5E5E5", edgecolor="white", lw=0.45,
                             ax=ax, zorder=3, alpha=0.95)
                non_geo.plot(facecolor="none", edgecolor="#888", hatch="///",
                             lw=0.25, ax=ax, zorder=4, alpha=0.5)

        boundary.plot(ax=ax, color="#111", lw=1.1, zorder=5)

        # Label extreme states
        if n_sig > 0:
            sig_geo = geo[sig_mask]
            top_sig = sig_geo.assign(_a=lambda d: d[coef_col].abs()) \
                              .sort_values("_a", ascending=False).head(5)
            label_states(ax, top_sig, fontsize=7)
        else:
            top_ext = geo.assign(_a=lambda d: d[coef_col].abs()) \
                          .sort_values("_a", ascending=False).head(4)
            label_states(ax, top_ext, fontsize=6.5, color="#555")

        sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm); sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, shrink=0.55, pad=0.02, aspect=22)
        cbar.set_label("Local β  (z-scored predictor)", fontsize=8)
        cbar.ax.tick_params(labelsize=7.5)
        cbar.outline.set_linewidth(0.5); cbar.outline.set_edgecolor("#999")

        add_north_arrow(ax); add_scale_bar(ax); style_map_axes(ax)

        bw_idx = {"b_urban":1, "b_htr":2, "b_edu":3, "b_wlth":4}[coef_col]
        bw    = int(mgwr_bws[bw_idx])
        if n_sig > 0:
            sub = f"{n_sig}/{len(geo)} states significant (α=0.05, Bonferroni-adjusted) · bandwidth = {bw} neighbours"
        else:
            sub = f"No state reaches significance after Bonferroni adjustment · bandwidth = {bw} neighbours"
        ax.set_title(f"{name}\n{sub}",
                     fontsize=10.5, fontweight="bold", pad=8, color="#1F3864")

    fig, axes = plt.subplots(2, 2, figsize=(15, 14), facecolor="white")
    specs = [
        ("b_urban","sig_urban", EQUITY_LABELS["urbanicity"]),
        ("b_htr",  "sig_htr",   EQUITY_LABELS["htr_distance"]),
        ("b_edu",  "sig_edu",   EQUITY_LABELS["no_education"]),
        ("b_wlth", "sig_wlth",  EQUITY_LABELS["low_wealth"]),
    ]
    for ax, (c, s, n) in zip(axes.flat, specs):
        _draw_coef_panel(ax, c, s, n)

    legend_handles = [
        mpatches.Patch(facecolor="#D73027", edgecolor="white",
                       label="β > 0   higher zero-dose risk"),
        mpatches.Patch(facecolor="#4575B4", edgecolor="white",
                       label="β < 0   lower zero-dose risk"),
        mpatches.Patch(facecolor="#E5E5E5", edgecolor="#888", hatch="///",
                       label="Not statistically significant"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3,
                fontsize=10, frameon=True, framealpha=0.95, edgecolor="#CCC",
                bbox_to_anchor=(0.5, 0.025))

    fig.suptitle("Multiscale Geographically Weighted Regression — spatial coefficient surfaces\n"
                 "Effect of each equity dimension on zero-dose risk by state, Nigeria (NDHS 2024)",
                 fontsize=14, fontweight="bold", y=0.99, color="#1F3864")
    fig.text(0.5, 0.005,
              "Boundaries: GRID3 Nigeria admin1 (vaccination layer) and admin2 (LGAs, faint). "
              f"MGWR R² = {mgwr_model.R2:.3f}. "
              "Adaptive bisquare kernel with AICc bandwidth selection. "
              "Projection: WGS84 / EPSG:4326.",
              ha="center", fontsize=7.5, color="#666", style="italic")
    plt.tight_layout(rect=[0, 0.055, 1, 0.965])

else:
    # Fallback scatter version
    fig, axes = plt.subplots(2, 2, figsize=(13, 12), facecolor=BG)
    for ax, (coef_col, sig_col, title) in zip(axes.flat, [
        ("b_urban","sig_urban","Urbanicity coefficient"),
        ("b_htr","sig_htr","HTR distance coefficient"),
        ("b_edu","sig_edu","Maternal no-education coefficient"),
        ("b_wlth","sig_wlth","Lowest-wealth coefficient")]):
        values = mgwr_full[coef_col].values
        sig    = mgwr_full[sig_col].values.astype(bool)
        vmax   = max(abs(values.min()), abs(values.max()))
        norm   = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        sc = ax.scatter(mgwr_full["lon"], mgwr_full["lat"],
                          c=values, cmap="RdBu_r", norm=norm,
                          s=350, edgecolor="black", lw=0.6, alpha=0.92)
        if (~sig).any():
            ax.scatter(mgwr_full.loc[~sig,"lon"], mgwr_full.loc[~sig,"lat"],
                       marker="x", s=80, color="#222", lw=1.5)
        plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02,
                     label="Coefficient (z-scored predictor)")
        for _, r in mgwr_full.iterrows():
            ax.annotate(r["state"], (r["lon"], r["lat"]),
                         textcoords="offset points", xytext=(5,5),
                         fontsize=6.5, color="#222")
        ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
        ax.set_facecolor(BG)
    fig.suptitle("MGWR coefficient surfaces (centroid view)",
                 fontsize=12, fontweight="bold", y=1.0)
    plt.tight_layout()

plt.savefig(f"{OUTPUT_DIR}/fig02_mgwr_coefficient_maps.png",
            dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig02_mgwr_coefficient_maps.png")


Saved: fig02_mgwr_coefficient_maps.png


In [17]:
# Figure 3 — MGWR bandwidth diagnostics
n_obs_local = len(df)
bw_df = pd.DataFrame({
    "predictor":  ["Intercept"] + [EQUITY_LABELS[v] for v in EQUITY_VARS],
    "bandwidth":  [int(b) for b in mgwr_bws],
})
print(bw_df.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4.8), facecolor="white")
y_pos = np.arange(len(bw_df))
bar_colors = ["#999" if b >= n_obs_local - 2
              else "#4C72B0" if b >= n_obs_local * 0.7
              else "#D73027"
              for b in bw_df["bandwidth"]]
ax.barh(y_pos, bw_df["bandwidth"], color=bar_colors,
        edgecolor="white", lw=0.6, height=0.6)
for i, b in enumerate(bw_df["bandwidth"]):
    if b >= n_obs_local - 2:
        lab = "essentially global"
    elif b >= n_obs_local * 0.7:
        lab = "near-global variation"
    else:
        lab = "localised variation"
    ax.text(b + 0.4, i, f"  {b}  ({lab})",
            va="center", fontsize=9.5, color="#222")
ax.set_yticks(y_pos); ax.set_yticklabels(bw_df["predictor"], fontsize=10)
ax.set_xlim(0, n_obs_local + 7)
ax.axvline(n_obs_local * 0.7, color="#666", ls=":", lw=0.8, alpha=0.6)
ax.text(n_obs_local * 0.7 + 0.3, len(bw_df) - 0.4,
        "70% of n", fontsize=7.5, color="#666",
        style="italic", ha="left", va="top")
ax.set_xlabel(f"Adaptive bandwidth (nearest-neighbour count, out of n = {n_obs_local})",
              fontsize=10)
ax.set_title("MGWR per-predictor bandwidths (AICc-optimal)\n"
              "Smaller bandwidth = the coefficient varies at a finer spatial scale",
              fontsize=12, fontweight="bold", pad=10, color="#1F3864")
ax.grid(axis="x", alpha=0.20, ls="--")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.set_facecolor("#FAFAFA")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig03_mgwr_bandwidths.png",
            dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig03_mgwr_bandwidths.png")


                                predictor  bandwidth
                                Intercept         35
                     Urbanicity (% urban)         24
HTR distance (% reporting access problem)         35
                Maternal no-education (%)         28
               Lowest wealth quintile (%)         27
Saved: fig03_mgwr_bandwidths.png


## 3. Approach B — Composite equity-deprivation index

### Why this index

Approach A (MGWR) estimates the local effect of each equity dimension — it answers *where* a dimension matters. The composite index answers a different, more operational question: *which states have the worst combined equity profile, and how should they be ranked for resource allocation?* A single score per state is easier to communicate and assign to programmatic tiers than a four-dimensional vector of locally-varying coefficients.

### Index construction

For each equity dimension:

1. **Sign-orient** so that higher values mean more deprivation. Urbanicity is the only dimension that requires flipping — high urbanicity is associated with lower zero-dose risk, so we use `s_rural = minmax(−urbanicity)`.
2. **Min-max rescale** each dimension to a 0–100 range: `s = (x − x_min) / (x_max − x_min) × 100`.
3. **Equal-weighted average**: `index = 0.25·s_rural + 0.25·s_htr + 0.25·s_edu + 0.25·s_wealth`.

### Why equal weights

Equal weighting is the default in the absence of strong prior information on relative importance. Data-driven weights (PCA loadings or regression coefficients on zero-dose) make the index a function of the outcome it is meant to predict from — circular by construction. The weights are exposed as configuration constants so sensitivity to alternative weights can be tested by changing one line.

### Tier classification

States are classified into four tiers using fixed cutpoints (25 / 50 / 75) on the composite. Fixed cutpoints keep tier labels stable across reruns — a state in Tier 1 today remains in Tier 1 unless its underlying indicators change.


In [18]:
# Sign-orient each dimension so higher score = more deprivation
df["s_rural"]   = minmax_scale(-df["urbanicity"])     # rural-share component
df["s_htr"]     = minmax_scale(df["htr_distance"])
df["s_noedu"]   = minmax_scale(df["no_education"])
df["s_lowwlth"] = minmax_scale(df["low_wealth"])

df["equity_index"] = (W_RURAL * df["s_rural"]
                      + W_HTR  * df["s_htr"]
                      + W_EDU  * df["s_noedu"]
                      + W_WLTH * df["s_lowwlth"])

df["equity_rank"] = df["equity_index"].rank(ascending=False, method="min").astype(int)
df["equity_tier"] = pd.cut(df["equity_index"],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=["Tier 4: Most equitable","Tier 3: Moderate",
            "Tier 2: Disadvantaged","Tier 1: Most disadvantaged"])

print("Tier composition:")
print(df["equity_tier"].value_counts().sort_index().to_string())

print("\nTop 10 most-disadvantaged states (composite equity-deprivation index):")
disp_cols = ["equity_rank","state","zone","zd_rate","equity_index","equity_tier"]
print(df.sort_values("equity_rank")[disp_cols].head(10).round(2).to_string(index=False))


Tier composition:
equity_tier
Tier 4: Most equitable         8
Tier 3: Moderate              20
Tier 2: Disadvantaged          8
Tier 1: Most disadvantaged     1

Top 10 most-disadvantaged states (composite equity-deprivation index):
 equity_rank   state          zone  zd_rate  equity_index                equity_tier
           1  Jigawa    North West     34.3         82.78 Tier 1: Most disadvantaged
           2  Sokoto    North West     86.5         70.41      Tier 2: Disadvantaged
           3   Kebbi    North West     84.0         66.58      Tier 2: Disadvantaged
           4  Kaduna    North West     45.6         58.25      Tier 2: Disadvantaged
           5 Adamawa    North East     29.1         55.50      Tier 2: Disadvantaged
           6   Niger North Central     56.0         53.83      Tier 2: Disadvantaged
           7  Ebonyi    South East      2.6         53.53      Tier 2: Disadvantaged
           8  Bauchi    North East     36.9         52.94      Tier 2: Disadvantaged
 

In [19]:
# Figure 4 — Composite equity-deprivation tier classification
# When polygons are available: choropleth + ranking side-by-side.
# When not: just the ranking bar.

if GEO_AVAILABLE:
    # Attach tier classification onto the geo dataframe
    geo = geo.merge(df[["state","equity_index","equity_tier","equity_rank"]],
                    on="state", how="left", suffixes=("","_d"))

    fig = plt.figure(figsize=(16, 11), facecolor="white")
    gs  = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.05)
    ax_map  = fig.add_subplot(gs[0,0])
    ax_rank = fig.add_subplot(gs[0,1])

    # ── Map panel ──
    xmin, ymin, xmax, ymax = geo.total_bounds
    pad = 0.5
    ax_map.set_xlim(xmin - pad, xmax + pad)
    ax_map.set_ylim(ymin - pad, ymax + pad + 0.3)
    ax_map.set_facecolor("#E8EFF5")
    if gdf_lga is not None:
        gdf_lga.plot(ax=ax_map, facecolor="none", edgecolor="#888",
                      lw=0.13, alpha=0.30, zorder=2)
    geo["_tcolor"] = geo["equity_tier"].astype(str).map(TIER_COLORS)
    geo.plot(color=geo["_tcolor"], ax=ax_map, edgecolor="white",
              lw=0.55, zorder=3)
    boundary.plot(ax=ax_map, color="#111", lw=1.2, zorder=5)

    priority = geo[geo["equity_tier"].astype(str)
                    .isin(["Tier 1: Most disadvantaged", "Tier 2: Disadvantaged"])]
    label_states(ax_map, priority, fontsize=7.5)

    add_north_arrow(ax_map); add_scale_bar(ax_map); style_map_axes(ax_map)
    ax_map.set_title("Composite equity-deprivation tier by state\n"
                      "Rural × HTR distance × Maternal no-education × Lowest wealth",
                      fontsize=12, fontweight="bold", pad=10, color="#1F3864")
    tier_handles = [mpatches.Patch(color=v, label=k) for k, v in TIER_COLORS.items()]
    ax_map.legend(handles=tier_handles, loc="lower right",
                   fontsize=9, framealpha=0.95, edgecolor="#CCC")

    # ── Ranking panel ──
    _d = df.sort_values("equity_index", ascending=True).reset_index(drop=True)
    y_p = np.arange(len(_d))
    col = [TIER_COLORS.get(str(t),"#888") for t in _d["equity_tier"]]
    ax_rank.set_facecolor("#FAFAFA")
    ax_rank.barh(y_p, _d["equity_index"], color=col,
                  edgecolor="white", lw=0.5, alpha=0.95, height=0.72)
    for i, r in _d.iterrows():
        ax_rank.text(r["equity_index"] + 0.8, i, f"  ZD={r['zd_rate']:.0f}%",
                      va="center", fontsize=8, color="#222")
    ax_rank.set_yticks(y_p); ax_rank.set_yticklabels(_d["state"], fontsize=9)
    ax_rank.set_xlim(0, 100)
    ax_rank.set_xlabel("Composite equity-deprivation index  (0 = least, 100 = most deprived)",
                        fontsize=10)
    ax_rank.set_title("State ranking", fontsize=12, fontweight="bold",
                       color="#1F3864", pad=10)
    ax_rank.grid(axis="x", alpha=0.20, ls="--")
    ax_rank.spines["top"].set_visible(False); ax_rank.spines["right"].set_visible(False)

    fig.suptitle("Composite equity-deprivation index — state classification, Nigeria 2024",
                 fontsize=14, fontweight="bold", y=0.985, color="#1F3864")
    fig.text(0.5, 0.005,
              "Boundaries: GRID3 Nigeria admin1.  Composite index: equal weights on rural, HTR, education, wealth.  "
              "Tier cutpoints: 25 / 50 / 75 on 0–100 composite.",
              ha="center", fontsize=8, color="#666", style="italic")
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
else:
    # Ranking-only fallback
    _d = df.sort_values("equity_index", ascending=True).reset_index(drop=True)
    y_p = np.arange(len(_d))
    col = [TIER_COLORS.get(str(t),"#888") for t in _d["equity_tier"]]
    fig, ax = plt.subplots(figsize=(11, 12), facecolor=BG)
    ax.set_facecolor(BG)
    ax.barh(y_p, _d["equity_index"], color=col, edgecolor="white",
            lw=0.5, alpha=0.93, height=0.72)
    for i, r in _d.iterrows():
        ax.text(r["equity_index"] + 0.8, i, f"  ZD={r['zd_rate']:.0f}%",
                va="center", fontsize=8.5, color="#222")
    ax.set_yticks(y_p); ax.set_yticklabels(_d["state"], fontsize=9.5)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Composite equity-deprivation index (0 = least, 100 = most deprived)",
                  fontsize=11)
    ax.set_title("State ranking by composite equity-deprivation index",
                  fontsize=12, fontweight="bold", pad=10)
    ax.grid(axis="x", alpha=0.2, ls="--")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    tier_h = [mpatches.Patch(color=v, label=k) for k, v in TIER_COLORS.items()]
    ax.legend(handles=tier_h, loc="lower right", fontsize=9)
    plt.tight_layout()

plt.savefig(f"{OUTPUT_DIR}/fig04_tier_classification_map.png",
            dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig04_tier_classification_map.png")


Saved: fig04_tier_classification_map.png


## 4. Joint synthesis — local effects within each equity tier

The MGWR model yields a predicted zero-dose rate for each state, and a coefficient for each equity dimension at each state. Aggregating these by the tiers from Approach B yields a tier-level summary that combines *what we measured* (observed zero-dose) with *what the model predicted* (MGWR fitted values) and *the dominant local drivers* (which coefficients are large within each tier).


In [20]:
# MGWR predicted zero-dose per state attached to the main dataframe
df["zd_pred"]  = mgwr_model.predy.ravel()
df["zd_resid"] = df["zd_rate"] - df["zd_pred"]
# Attach MGWR coefficients
for c in coef_cols:
    df[c] = mgwr_coef[c].values

# Tier-level summary
tier_rows = []
for tier, sub in df.groupby("equity_tier", observed=True):
    tier_rows.append({
        "equity_tier":  str(tier),
        "n_states":     len(sub),
        "obs_mean":     sub["zd_rate"].mean(),
        "pred_mean":    sub["zd_pred"].mean(),
        "mean_b_urban": sub["b_urban"].mean(),
        "mean_b_htr":   sub["b_htr"].mean(),
        "mean_b_edu":   sub["b_edu"].mean(),
        "mean_b_wlth":  sub["b_wlth"].mean(),
    })
tier_tab = pd.DataFrame(tier_rows).sort_values("pred_mean", ascending=False)
print("Tier-level summary:")
print(tier_tab.round(2).to_string(index=False))
tier_tab.to_csv(f"{OUTPUT_DIR}/equity_tier_summary.csv", index=False)

# Figure 5 — Predicted vs observed by tier (premium paired bars)
_tt   = tier_tab.sort_values("pred_mean", ascending=True).reset_index(drop=True)
y_pos = np.arange(len(_tt))
col   = [TIER_COLORS.get(t, "#888") for t in _tt["equity_tier"]]

fig, ax = plt.subplots(figsize=(10.5, 5.5), facecolor="white")
ax.set_facecolor("#FAFAFA")
h = 0.34
ax.barh(y_pos - h/2, _tt["pred_mean"], height=h, color=col,
        edgecolor="white", lw=0.5, alpha=0.95, label="MGWR predicted")
ax.barh(y_pos + h/2, _tt["obs_mean"], height=h, color=col,
        edgecolor="white", lw=0.5, alpha=0.55, hatch="///",
        label="Observed (NDHS 2024)")

for i, r in _tt.iterrows():
    ax.text(max(r["pred_mean"], r["obs_mean"]) + 1.0, i,
            f"  pred {r['pred_mean']:.1f}%   obs {r['obs_mean']:.1f}%   (n={int(r['n_states'])})",
            va="center", fontsize=9.5, color="#222")

ax.set_yticks(y_pos); ax.set_yticklabels(_tt["equity_tier"], fontsize=10)
xmax = max(_tt["pred_mean"].max(), _tt["obs_mean"].max()) * 1.45
ax.set_xlim(0, xmax)
ax.set_xlabel("Zero-dose rate (%)", fontsize=10)
ax.grid(axis="x", alpha=0.20, ls="--")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.legend(loc="lower right", fontsize=9.5, framealpha=0.95, edgecolor="#CCC")
ax.set_title("Predicted vs observed zero-dose rate by equity-deprivation tier\n"
              "MGWR posterior mean (solid) and NDHS 2024 observed (hatched)",
              fontsize=12, fontweight="bold", pad=10, color="#1F3864")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig05_predicted_vs_observed_by_tier.png",
            dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig05_predicted_vs_observed_by_tier.png")


Tier-level summary:
               equity_tier  n_states  obs_mean  pred_mean  mean_b_urban  mean_b_htr  mean_b_edu  mean_b_wlth
     Tier 2: Disadvantaged         8     47.00      48.58         -3.42        0.49       14.68        -1.25
Tier 1: Most disadvantaged         1     34.30      44.90         -1.06        0.45       11.88        -1.62
          Tier 3: Moderate        20     28.07      27.36         -2.74        0.44       16.70        -2.30
    Tier 4: Most equitable         8     15.09      14.37         -4.41        0.16       18.18        -3.03
Saved: fig05_predicted_vs_observed_by_tier.png


## 5. Limitations and caveats

A short list of constraints to accompany any external presentation of these results.

- **Granularity.** The analysis is at state level (n = 37). The four equity dimensions all have substantial within-state heterogeneity that is invisible at this resolution. LGA- or ward-level analysis is the natural next step once disaggregated microdata are available; MGWR scales directly to that resolution.
- **Sample size for the AICc-MGWR.** With n = 37 the Monte Carlo test for non-stationarity of individual coefficients is unstable; we rely instead on the Bonferroni-style local t-statistics, which are well-defined for this n.
- **Centroids vs polygons.** When GRID3 admin1 shapefiles are not uploaded, the maps use state centroids and point symbols. The MGWR fit itself is identical in either case — only the visualisation changes.
- **Equal weighting in the composite.** Sensitivity to alternative weights was not tested in this iteration. Single-dimension specifications would reveal which tier assignments are robust to weighting choice.
- **Cross-sectional snapshot.** The analysis uses NDHS 2024 only. Trend-aware extensions across NDHS 2008, 2013, 2018, and 2024 would identify states whose equity profile is improving or worsening over time.


## 6. Outputs

The final state-level table joins observed zero-dose, MGWR-predicted zero-dose, the per-state MGWR coefficients with significance flags, the four standardised equity scores, the composite index, and the tier classification.


In [21]:
out_cols = [
    "state","zone","lat","lon",
    "zd_rate","zd_pred","zd_resid",
    "urbanicity","htr_distance","no_education","low_wealth",
    "s_rural","s_htr","s_noedu","s_lowwlth",
    "equity_index","equity_rank","equity_tier",
    "intercept","b_urban","b_htr","b_edu","b_wlth",
]
final = df[out_cols].sort_values("equity_rank").reset_index(drop=True)
final_path = f"{OUTPUT_DIR}/equity_state_rankings.csv"
final.to_csv(final_path, index=False)
print(f"Saved: {final_path}  ({len(final)} states)")

print("\nTop 5 most-disadvantaged states (final output):")
print(final.head(5)[["equity_rank","state","zone","zd_rate","zd_pred",
                      "equity_index","equity_tier","b_edu"]]
              .round(2).to_string(index=False))


Saved: outputs/equity_state_rankings.csv  (37 states)

Top 5 most-disadvantaged states (final output):
 equity_rank   state       zone  zd_rate  zd_pred  equity_index                equity_tier  b_edu
           1  Jigawa North West     34.3    44.90         82.78 Tier 1: Most disadvantaged  11.88
           2  Sokoto North West     86.5    79.87         70.41      Tier 2: Disadvantaged  17.54
           3   Kebbi North West     84.0    82.46         66.58      Tier 2: Disadvantaged  18.09
           4  Kaduna North West     45.6    38.32         58.25      Tier 2: Disadvantaged  14.57
           5 Adamawa North East     29.1    31.60         55.50      Tier 2: Disadvantaged   8.83


---

In [22]:
# ─── COLAB: Download all output files ─────────────────────────────────
if IN_COLAB:
    from google.colab import files
    import glob
    for fp in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
        files.download(fp)
    print("Download prompts issued.")
else:
    import glob
    print("Files in outputs/:")
    for fp in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
        print(f"  {fp}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download prompts issued.


In [ ]:
# ============================================================================
# [CIDRE-Quantium] CSV table-formatting pass  -  appended for GAVI review
# ----------------------------------------------------------------------------
# Purpose : make this notebook's GENERATED CSV tables presentation-ready.
#           (consistent rounding to 4 dp, integer count/rank columns, and a
#            styled .xlsx copy with a bold frozen header).
# Scope   : ONLY this notebook's own result tables, listed in _FMT_TABLES below.
#           Input/source data files are never touched.
# Safety  : runs AFTER all analysis cells; does not change any analysis or any
#           column names / row counts, so end-to-end behaviour is unchanged.
#           Every step is guarded - it can never raise and stop the notebook.
# ============================================================================
import os as _os
import numpy as _np
import pandas as _pd

try:
    _FMT_DIR = str(OUTPUT_DIR)            # this notebook's own output directory
except NameError:
    _FMT_DIR = "outputs"               # fallback if the cell is run in isolation

_FMT_TABLES = ["equity_tier_summary.csv", "equity_state_rankings.csv", "equity_model_comparison.csv", "equity_mgwr_state_coefficients.csv"]
_INT_HINTS  = ("count", "rank", "year", "yr", "pop", "cohort",
               "n_children", "susceptible", "_n")

def _fmt_one(_path):
    if not _os.path.exists(_path):
        return False
    try:
        _df = _pd.read_csv(_path)
    except Exception as _e:
        print("[format] skip", _os.path.basename(_path), "->", _e)
        return False
    for _col in _df.select_dtypes(include=[_np.number]).columns:
        _name = str(_col).lower(); _s = _df[_col]
        if any(_h in _name for _h in _INT_HINTS) and _s.dropna().mod(1).eq(0).all():
            _df[_col] = _s.astype("Int64")          # tidy whole-number counts
        else:
            _df[_col] = _s.round(4)                 # consistent decimals
    _df.to_csv(_path, index=False)                  # same schema, tidier values
    try:                                            # styled Excel companion
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
        _xp = _os.path.splitext(_path)[0] + ".xlsx"
        with _pd.ExcelWriter(_xp, engine="openpyxl") as _xw:
            _df.to_excel(_xw, index=False, sheet_name="Table")
            _ws = _xw.sheets["Table"]
            _fill = PatternFill("solid", fgColor="1F3B57")
            _font = Font(bold=True, color="FFFFFF")
            _thin = Side(style="thin", color="D9D9D9")
            _bd = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
            for _j, _c in enumerate(_df.columns, 1):
                _cell = _ws.cell(row=1, column=_j)
                _cell.fill = _fill; _cell.font = _font
                _cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                _ws.column_dimensions[get_column_letter(_j)].width = min(max(len(str(_c)) + 2, 10), 40)
            for _i in range(2, len(_df) + 2):
                for _j in range(1, len(_df.columns) + 1):
                    _ws.cell(row=_i, column=_j).border = _bd
            _ws.freeze_panes = "A2"; _ws.row_dimensions[1].height = 26
    except Exception as _e:
        print("[format] xlsx styling skipped for", _os.path.basename(_path), "->", _e)
    return True

_done = [_t for _t in _FMT_TABLES if _fmt_one(_os.path.join(_FMT_DIR, _t))]
if _FMT_TABLES:
    print("Table formatting complete -", len(_done), "of", len(_FMT_TABLES),
          "result table(s) formatted in '%s':" % _FMT_DIR)
    for _t in _done:
        print("   -", _t)
else:
    print("Domain produces figure/report outputs only - no result CSV tables to format.")
